<a href="https://colab.research.google.com/github/sarshadad-codeee/FlyRank_ML_Task1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())

!pip install duckdb --quiet

Working directory: /content/FlyRank_ML_Task1/FlyRank_ML_Task1


In [4]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")  # must already be set as a Colab Secret

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"

In [5]:
#main fact table schema
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 1
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [6]:
#dim_content schema
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{base}/dim_content.parquet') LIMIT 1").show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [7]:
#dim_clients schema
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{base}/dim_clients.parquet') LIMIT 1").show()

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_gsc_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_ga4_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ access_profile      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_created_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_updated_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_start      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_start      │ DATE        │ YES     │ NULL    │ NULL  

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
"""One row = one content item (content_hash_id) belonging to one client
(client_hash_id), summarized over a fixed trailing month — not the raw
daily grain the fact table ships in. I aggregate
fact_content_daily_performance up to content-per-month before scoring.

Time window: one mid-panel month at a time (month=2026-03 for this
notebook). For a real model I would compare a prior month's aggregated
signals against the following month's outcome — never the final sealed
month (June 2026, the _sample table)."""

"One row = one content item (content_hash_id) belonging to one client\n(client_hash_id), summarized over a fixed trailing month — not the raw\ndaily grain the fact table ships in. I aggregate\nfact_content_daily_performance up to content-per-month before scoring.\n\nTime window: one mid-panel month at a time (month=2026-03 for this\nnotebook). For a real model I would compare a prior month's aggregated\nsignals against the following month's outcome — never the final sealed\nmonth (June 2026, the _sample table)."

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
"""**Feature:** gsc_impressions, gsc_clicks, gsc_sum_position (→ average
position), sessions_ai, scroll_events, word_count, char_count,
category_count — all knowable at or before the decision moment.

**Label / proxy:** month-over-month change in gsc_clicks (this month's
clicks vs. previous month's clicks) — provisional decline proxy.

**Context (never features):** content_hash_id, client_hash_id, month,
report_date — used only for grouping/joining/filtering.

**Excluded:** last_optimized_date, optimization_eligible_date — these
reflect a past human/product decision about which pages were already
chosen for attention; using them would leak editorial judgment into
what should be a fresh, data-driven priority score. Also excluding
provider_used and model_used (AI-content-generation metadata, not a
performance signal for this lane)."""

"**Feature:** gsc_impressions, gsc_clicks, gsc_sum_position (→ average\nposition), sessions_ai, scroll_events, word_count, char_count,\ncategory_count — all knowable at or before the decision moment.\n\n**Label / proxy:** month-over-month change in gsc_clicks (this month's\nclicks vs. previous month's clicks) — provisional decline proxy.\n\n**Context (never features):** content_hash_id, client_hash_id, month,\nreport_date — used only for grouping/joining/filtering.\n\n**Excluded:** last_optimized_date, optimization_eligible_date — these\nreflect a past human/product decision about which pages were already\nchosen for attention; using them would leak editorial judgment into\nwhat should be a fresh, data-driven priority score. Also excluding\nprovider_used and model_used (AI-content-generation metadata, not a\nperformance signal for this lane)."

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
month_path = f"{base}/fact_content_daily_performance/month=2026-03/*.parquet"
print(month_path)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


In [11]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{month_path}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



In [12]:
#Query 2 — Row count + date span
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS earliest,
           MAX(report_date) AS latest,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM read_parquet('{month_path}')
""").show()

┌────────────┬────────────┬────────────┬───────────┬─────────────────┐
│ total_rows │  earliest  │   latest   │ n_clients │ n_content_items │
│   int64    │    date    │    date    │   int64   │      int64      │
├────────────┼────────────┼────────────┼───────────┼─────────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │        55 │          331437 │
└────────────┴────────────┴────────────┴───────────┴─────────────────┘



In [13]:
#Query 3 — Availability, filtered with IS TRUE:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{month_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int128       │       int128       │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



"""
Query results confirm the contract:

Grain check: 0 rows returned when grouping by report_date + client_hash_id
+ content_hash_id and filtering for duplicates — the grain holds. One row
really is one report_date x client x content item.

Row count + date span: 9,841,378 rows for month=2026-03, spanning the full
month (2026-03-01 to 2026-03-31), across 55 distinct clients and 331,437
distinct content items. Note: 55 clients here vs. 104 total in dim_clients
— roughly half the client base has zero activity rows in this specific
month, consistent with the panel's unbalanced-history warning.

Availability: of 9,841,378 total rows, only 3,611,061 (36.7%) have
gsc_data_available = TRUE, and only 413,966 (4.2%) have ga4_data_available
= TRUE. This is a large, deliberate filter -- most rows in the raw fact
table are zero-filled placeholders for dates before a client's own
gsc_data_start / ga4_data_start, not real zero-activity days. Any feature
build for this lane must filter on these flags first, or it will silently
average in fake zeros as if they were real measurements.
"""

In [14]:
features_df = con.sql(f"""
    WITH gsc_part AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_clicks) AS total_clicks,
            SUM(gsc_impressions) AS total_impressions,
            AVG(gsc_sum_position) AS avg_position_proxy
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    ),
    ga4_part AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(sessions_ai) AS total_ai_sessions,
            SUM(scroll_events) AS total_scroll_events
        FROM read_parquet('{month_path}')
        WHERE ga4_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT *
    FROM gsc_part
    LEFT JOIN ga4_part USING (content_hash_id, client_hash_id)
""").df()

print(f"Feature frame shape: {features_df.shape}")
print(f"Rows with real GA4 data: {features_df['total_ai_sessions'].notna().sum()}")
features_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)
Rows with real GA4 data: 71888


,content_hash_id,client_hash_id,total_clicks,total_impressions,avg_position_proxy,total_ai_sessions,total_scroll_events
0,content_1d69c2ed06358f6f,client_9958f0a7ae1df715,0.0,63.0,39.315789,0.0,5.0
1,content_1886b144fd43edc5,client_9958f0a7ae1df715,3.0,2289.0,426.032258,0.0,7.0
2,content_f6ad9abe97a3a429,client_9958f0a7ae1df715,0.0,223.0,58.322581,0.0,1.0
3,content_9b3decff0d7e6690,client_9958f0a7ae1df715,1.0,630.0,253.516129,0.0,0.0
4,content_55ead56c1217a888,client_9958f0a7ae1df715,3.0,347.0,135.677419,0.0,1.0
5,content_183fac80c092efab,client_9958f0a7ae1df715,0.0,140.0,124.935484,0.0,3.0
6,content_ece73a0cde53ecee,client_9958f0a7ae1df715,1.0,336.0,176.866667,0.0,5.0
7,content_b813c73d7000b3b1,client_9958f0a7ae1df715,3.0,720.0,212.129032,0.0,5.0
8,content_79d8051bbc47e80b,client_9958f0a7ae1df715,0.0,211.0,230.129032,0.0,1.0
9,content_5a368b854dd375b2,client_9958f0a7ae1df715,1.0,3877.0,905.032258,0.0,2.0


"knowable at decision moment" justification
"""
Data-quality catch during feature building: the first attempt used a single
gsc_data_available filter for ALL five features, but total_ai_sessions and
total_scroll_events come from GA4, not GSC -- they showed 100% NaN under
that filter. Fixed by aggregating GSC-sourced and GA4-sourced features
separately, each filtered on its own availability flag, then joining.

Of 176,738 content-client rows with real GSC data this month, only 71,888
(40.7%) also have real GA4 data -- confirming the skill file's warning that
a large share of clients/content have GSC-only history. Any feature that
mixes both sources must account for this gap explicitly (e.g. via a
has_ga4_data flag) rather than silently treating missing GA4 rows as zero
engagement.
"""

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Build a simple proxy label: "high engagement" content (above median clicks)
features_df["is_high_engagement"] = (
    features_df["total_clicks"] > features_df["total_clicks"].median()
).astype(int)

X_cols = ["total_impressions", "avg_position_proxy", "total_ai_sessions", "total_scroll_events"]
X = features_df[X_cols].fillna(0)
y = features_df["is_high_engagement"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)
honest_score = accuracy_score(y_test, clf.predict(X_test))
print(f"Honest baseline accuracy (no leak): {honest_score:.3f}")

Honest baseline accuracy (no leak): 0.840


In [16]:
#Add the deliberate leak
# THE TRAP: deliberately add a label-derived column as a "feature"
# total_clicks was literally used to CREATE the label (is_high_engagement),
# so including it is pure leakage — the model would just be reading the answer.

X_leaky_cols = X_cols + ["total_clicks"]  # <-- the deliberate leak
X_leaky = features_df[X_leaky_cols].fillna(0)

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaky, y, test_size=0.3, random_state=42
)

clf_leaky = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leaky.fit(X_train_leak, y_train_leak)
leaky_score = accuracy_score(y_test_leak, clf_leaky.predict(X_test_leak))

print(f"Honest baseline accuracy (no leak):  {honest_score:.3f}")
print(f"Leaky accuracy (with total_clicks):  {leaky_score:.3f}")
print(f"Jump: +{(leaky_score - honest_score):.3f}")

Honest baseline accuracy (no leak):  0.840
Leaky accuracy (with total_clicks):  1.000
Jump: +0.160


#Write up the lesson, then confirm you're keeping the honest number
"""
The trap, sprung on purpose: total_clicks was used to CREATE the label
(is_high_engagement = clicks > median), so adding total_clicks back in as
a "feature" let the model essentially read the answer directly off the
label's own source column.

Honest baseline (no leak):  {honest_score:.3f}
Leaky score (with the label-derived column): {leaky_score:.3f}

This is the same lesson as notebook 02's is_declining_label trap and Lane
2's trend_direction warning, now reproduced on real warehouse data by me:
any column used to construct the label -- or directly derived from it --
must be excluded as a feature, no matter how good it makes the score look.
The honest number, 0.840, is the one I'm keeping and reporting going
forward.
""".format(honest_score=honest_score, leaky_score=leaky_score)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

"""
Data limit: this month's slice (2026-03) only reflects 55 of the 104 total
clients in dim_clients -- roughly half the client base has no activity
rows in this specific month, consistent with each client's own
gsc_data_start/ga4_data_start rather than a shared calendar history. This
data can never tell you whether a client's absence in a given month means
"no content activity happened" versus "this client's tracking simply
hadn't started yet" -- that distinction requires checking dim_clients
directly, not just the fact table, before drawing any conclusion about a
missing client-month.
"""

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.